# 04 — Scaffold-constrained generation (colchicine site)

Sometimes you don't want any tubulin inhibitor — you want one that *keeps*
the colchicine-site scaffold (3,4,5-trimethoxyphenyl + biaryl + tropone).
This tutorial shows how to bias the GA so every candidate retains the
scaffold while exploring fragment substitutions everywhere else.

Pieces used:
- `extract_scaffold(mol)` — Bemis-Murcko-style scaffold via chython's `skin_graph`
- `tanimoto_distance_counts` — the diversity term (1 − Tanimoto)
- `decode_population`, `filter_molecule`         (chython side)
- `mol_to_rdkit`, `check_smarts_alerts`,
  `pains_brenk_nih_filter`                       (RDKit side)
- DEAP NSGA-II with two custom operators that respect a **fixed-fragment mask**

The seed population (51 ChEMBL hits sharing the colchicine scaffold) is
bundled at `data/tubulin/colchicine_scaffold.sdf`.

Heads-up: the published run did 200 NSGA-II generations. To keep this
tutorial tractable on a laptop we cap it at 5 — the patterns and outputs
are the same; bump `NUM_GENS` for paper-quality runs.


## Imports

In [ ]:
import os
import pickle
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from chython.files import SDFRead

from VQGAE import (
    VQGAE, OrderingNetwork,
    encode_smiles, decode_population,
    extract_scaffold, tanimoto_distance_counts,
    frag_inds_to_counts, frag_counts_to_inds,
    vqgae_encode_mols, filter_molecule,
    mol_to_rdkit, check_smarts_alerts, pains_brenk_nih_filter,
    smiles_to_mol,   # robust SMILES parser (chython + RDKit fallback)
)

## Define the scaffold (colchicine, 3,4,5-trimethoxyphenyl tropone)

In [ ]:
colchicine = smiles_to_mol("COc1cc2c(c(OC)c1OC)-c1ccc(OC)c(=O)cc1[C@@H](NC(C)=O)CC2")
colchicine.kekule()
colchicine_scaffold = extract_scaffold(colchicine)
colchicine_scaffold.thiele()
print(f"colchicine: {len(colchicine)} atoms; scaffold: {len(colchicine_scaffold)} atoms")
colchicine

## Load the bundled seed population (51 colchicine-scaffold ChEMBL hits)

In [ ]:
DATA = "../data/tubulin"
seed_mols = []
with SDFRead(f"{DATA}/colchicine_scaffold.sdf") as inp:
    for mol in inp:
        seed_mols.append(mol)
print(f"loaded {len(seed_mols)} seed molecules")

## Load checkpoints + RF (CPU; flip to cuda if you have a GPU)

In [ ]:
DEV = "cpu"  # set to 'cuda' on a GPU box
BATCH = 50
REPO = "tagirshin/VQGAE"
vqgae_enc = VQGAE.load_from_checkpoint(
    hf_hub_download(REPO, "vqgae.ckpt"), task="encode", batch_size=BATCH, map_location=DEV
).eval().to(DEV)
vqgae_dec = VQGAE.load_from_checkpoint(
    hf_hub_download(REPO, "vqgae.ckpt"), task="decode", batch_size=BATCH, map_location=DEV
).eval().to(DEV)
ordering_model = OrderingNetwork.load_from_checkpoint(
    hf_hub_download(REPO, "ordering_network.ckpt"), batch_size=BATCH, map_location=DEV
).eval().to(DEV)

with open(f"{DATA}/rf_class_train_tubulin.pickle", "rb") as fh:
    rf_model = pickle.load(fh)

## Build the fixed-fragment mask

For each seed molecule, identify which codebook fragments correspond to
**scaffold atoms**. The union of those positions is the mask the GA must
preserve.


In [ ]:
seed_inds = vqgae_encode_mols(seed_mols, vqgae_enc, batch_size=BATCH)
final_scaffold_mask = np.zeros(4096, dtype=np.int64)
for n, mol in enumerate(seed_mols):
    mol.thiele()
    mapping = list(colchicine_scaffold.get_mapping(mol))[0]
    scaffold_keys = np.array(sorted(mapping.values())) - 1
    scaffold_atoms = seed_inds[n][scaffold_keys].tolist()
    padded = np.array([scaffold_atoms + [-1] * (51 - len(scaffold_atoms))])
    final_scaffold_mask += np.where(frag_inds_to_counts(padded)[0] > 0, 1, 0)
final_scaffold_mask = (final_scaffold_mask > 0).astype(np.int64)
scaffold_size = len(colchicine_scaffold)
fixed_indices = set(np.where(final_scaffold_mask > 0)[0].tolist())
print(f"{final_scaffold_mask.sum()} fragment positions locked by the scaffold (of 4096)")
print(f"scaffold heavy-atom budget: {scaffold_size}")

## Multi-objective fitness (NSGA-II)

Five outputs (all maximised):
- `rf` — RF probability of activity, ∈ [0, 1]
- `novelty` — `min` Tanimoto distance to seeds (further from seeds = better)
- `ordering` — OrderingNetwork confidence, ∈ [0, 1]
- `size_penalty` — −1 if heavy-atom count < 18, else 0
- `scaffold_filter` — −2 if scaffold positions exceed budget, else 0


In [ ]:
initial_pop = frag_inds_to_counts(seed_inds)


def fitness_batch_multi(individual_array: np.ndarray) -> np.ndarray:
    counts = individual_array.copy()
    rf = rf_model.predict_proba(counts)[:, 1]
    scaff_counts = counts * final_scaffold_mask
    scaffold_filter = np.where(scaff_counts.sum(-1) - scaffold_size > 0, -2, 0)
    size_penalty = np.where(counts.sum(-1) < 18, -1.0, 0.0)
    dissim = tanimoto_distance_counts(counts, initial_pop).min(-1)
    novelty = dissim + np.where(dissim == 0, -5, 0)
    inds = frag_counts_to_inds(counts, max_atoms=51)
    from VQGAE import restore_order
    _, ord_scores = restore_order(inds, ordering_model)
    return np.column_stack((rf, novelty, np.array(ord_scores), size_penalty, scaffold_filter))

## DEAP NSGA-II with scaffold-aware crossover and mutation

Both operators take a `fixed_indices` set. Crossover never swaps positions in
that set; mutation never shuffles them. Everything else is fair game.


In [ ]:
from deap import base, creator, tools


def custom_one_point_crossover(ind1, ind2, fixed_indices):
    size = min(len(ind1), len(ind2))
    cxpoint = random.randint(1, size - 1)
    fixed_mask = np.zeros(size, dtype=bool)
    fixed_mask[list(fixed_indices)] = True
    temp1, temp2 = ind1.copy(), ind2.copy()
    ind1[cxpoint:] = np.where(fixed_mask[cxpoint:], ind1[cxpoint:], temp2[cxpoint:])
    ind2[cxpoint:] = np.where(fixed_mask[cxpoint:], ind2[cxpoint:], temp1[cxpoint:])
    return ind1, ind2


def custom_mut_shuffle_indexes(individual, fixed_indices, indpb):
    size = len(individual)
    non_fixed = [i for i in range(size) if i not in fixed_indices]
    to_shuffle = [i for i in non_fixed if random.random() < indpb]
    if len(to_shuffle) > 1:
        shuffled = individual[to_shuffle]
        np.random.shuffle(shuffled)
        individual[to_shuffle] = shuffled
    return individual,


creator.create("FitnessMulti", base.Fitness, weights=(0.5, 0.4, 0.2, 1.0, 1.0))
creator.create("Individual", np.ndarray, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()
toolbox.register("evaluate", fitness_batch_multi)
toolbox.register("mate", custom_one_point_crossover, fixed_indices=fixed_indices)
toolbox.register("mutate", custom_mut_shuffle_indexes,
                 fixed_indices=fixed_indices, indpb=0.2)
toolbox.register("select", tools.selNSGA2)

## Run NSGA-II (5 generations — bump to 200 for paper-quality)

In [ ]:
NUM_GENS = 5
EVAL_BATCH = 100
random.seed(42)


def evaluate(population, batch_size):
    pop_array = np.array(population)
    fitnesses = np.zeros((pop_array.shape[0], 5))
    for i in range(0, pop_array.shape[0], batch_size):
        batch = pop_array[i:i + batch_size]
        fitnesses[i:i + batch_size] = toolbox.evaluate(batch)
    for ind, fit in zip(population, fitnesses.tolist()):
        ind.fitness.values = tuple(fit)


pop = [creator.Individual(ind) for ind in initial_pop]
evaluate(pop, EVAL_BATCH)
all_solutions = [initial_pop[0].copy()]

for gen in tqdm(range(NUM_GENS), desc="NSGA-II"):
    offspring = list(map(toolbox.clone, toolbox.select(pop, len(pop))))
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < 0.5:
            toolbox.mate(child1, child2)
            del child1.fitness.values, child2.fitness.values
    for mutant in offspring:
        if random.random() < 0.05:
            toolbox.mutate(mutant)
            del mutant.fitness.values
    invalid = [ind for ind in offspring if not ind.fitness.valid]
    evaluate(invalid, EVAL_BATCH)
    pop[:] = offspring
    promising = [ind for ind in invalid if ind.fitness.values[0] > 0.5]
    if promising:
        all_solutions.append(np.unique(np.array(promising), axis=0))

solutions = np.unique(np.vstack(all_solutions), axis=0)
print(f"{solutions.shape[0]} unique candidates")

## Decode and apply the full filter cascade

In [ ]:
mols, validity, _ = decode_population(solutions, vqgae_dec, ordering_model, batch_size=BATCH)

passed: list = []
seed_set = set(seed_mols)
for mol, ok in zip(mols, validity):
    if not ok or mol in seed_set:
        continue
    if not filter_molecule(mol):
        continue
    try:
        rd = mol_to_rdkit(mol, compute_2d=False)
    except Exception:
        continue
    if not check_smarts_alerts(rd):
        continue
    if not pains_brenk_nih_filter(rd):
        continue
    # require the scaffold substructure on the survivor
    scaff = extract_scaffold(mol)
    scaff.thiele()
    if colchicine_scaffold <= scaff:
        passed.append(mol)
print(f"{len(passed)} survivors keep the colchicine scaffold + clean filters")

## Save and inspect

In [ ]:
out_path = "colchicine_scaffold_generated.smi"
with open(out_path, "w") as fh:
    for mol in passed:
        fh.write(f"{mol}\n")
print(f"wrote {len(passed)} SMILES to {out_path}")

# show 5 examples
for mol in passed[:5]:
    print(" ", mol)

## What's next

- Bump `NUM_GENS` to 200 and `EVAL_BATCH` to 500 for the paper run; expect
  ~30k unique candidates → ~600 post-filter survivors and ~60% of those
  retaining the colchicine scaffold (numbers from the published notebook).
- For a different scaffold, replace the colchicine SMILES at the top.
  `extract_scaffold` and `final_scaffold_mask` are scaffold-agnostic.
- Combine with tutorial 03's BO swap: the per-trial Optuna sampler can be
  given the same `fixed_indices` constraint as a soft penalty, eliminating
  the need for the DEAP NSGA-II loop.
